# XRS Reconstruction Performance Models

Hands-on examples for calling the model-backed phase reconstruction resource predictors directly and through SystemFlow mutations.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next(
    p for p in [cwd, *cwd.parents]
    if (p / "systemflow").is_dir()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from systemflow.auxtypes import Message
from systemflow.node import Component, ExecutionGraph
from systemflow.mutations import InputMessage
from systemflow.xrs import PhaseReconstruction2D_model, PhaseReconstruction3D_model
from systemflow.xrs_models import PtyChiResourceModel, PtychoPINNResourceModel

## Direct predictor calls

The predictors can be called without building a SystemFlow graph. Pty-Chi defaults to A100, dm, batch size 1000, and 500 iterations. PtychoPINN defaults to A100, batch size 1024, and grouped samples = 0.78 * raw images.

In [2]:
ptychi = PtyChiResourceModel.from_bundled_data()
ptychopinn = PtychoPINNResourceModel.from_bundled_data()

num_images = 1000
resolution = (64, 64)

ptychi_est = ptychi.predict(num_images=num_images, resolution=resolution)
pinn_est = ptychopinn.predict(num_images=num_images, resolution=resolution)

pd.DataFrame([
    {
        "backend": ptychi_est.metadata["backend"],
        "gpu": ptychi_est.metadata["gpu"],
        "algorithm": ptychi_est.metadata.get("algorithm", ""),
        "images": ptychi_est.metadata["num_images"],
        "latency_s": ptychi_est.latency_s,
        "energy_j": ptychi_est.energy_j,
        "avg_power_w": ptychi_est.avg_power_w,
        "images_per_joule": ptychi_est.images_per_joule,
    },
    {
        "backend": pinn_est.metadata["backend"],
        "gpu": pinn_est.metadata["gpu"],
        "algorithm": pinn_est.metadata.get("algorithm", ""),
        "images": pinn_est.metadata["num_images"],
        "latency_s": pinn_est.latency_s,
        "energy_j": pinn_est.energy_j,
        "avg_power_w": pinn_est.avg_power_w,
        "images_per_joule": pinn_est.images_per_joule,
    },
])

,backend,gpu,algorithm,images,latency_s,energy_j,avg_power_w,images_per_joule
0,ptychi,A100,dm,1000,26.268491,6986.303811,265.957557,0.143137
1,ptychopinn,A100,,1000,0.494500,16.691648,33.754596,59.910204


## 2D model-backed reconstruction mutation

This mirrors a BCDI-style single-pattern reconstruction. The resource model reads resolution and bit depth from the input message.

In [3]:
input_msg_2d = Message(
    fields={},
    properties={
        "resolution (n,n)": (64, 64),
        "bitdepth (n)": 16,
    },
)

analysis_2d = Component(
    "2D analysis",
    [InputMessage(), PhaseReconstruction2D_model()],
    {"input message (Message)": input_msg_2d},
)

g2d = ExecutionGraph("2D reconstruction performance", [analysis_2d], [])()
g2d.get_output_msg(), g2d.root_node.properties

(Message(fields={'phase data (B)': 8192.0}, properties={'resolution (n,n)': (64, 64), 'bitdepth (n)': 16, 'phase reconstruction (n,n)': (64, 64), 'phase reconstruction latency (s)': 0.6273140242624}),
 {'phase reconstruction ops (n,n)': 737279999.9999999,
  'phase reconstruction energy (J)': 80.99157025207892,
  'phase reconstruction power (W)': 129.10849609541145,
  'phase reconstruction images per joule (images/J)': 0.012346963972763964,
  'phase reconstruction io latency (s)': 0.37164737999999997,
  'phase reconstruction compute latency (s)': 0.2556666442624,
  'phase reconstruction io energy (J)': 12.04230423045,
  'phase reconstruction compute energy (J)': 68.94926602162892,
  'phase reconstruction metadata': {'backend': 'ptychi',
   'gpu': 'A100',
   'algorithm': 'dm',
   'num_images': 1,
   'resolution': (64, 64),
   'iterations': 500,
   'batch_size': 1000,
   'num_batches': 1,
   'total_gflops': 0.7372799999999999}})

## 3D ptychography reconstruction with Pty-Chi

For ptychography, the model-backed mutation uses `stored images (n)` as the image count. System-side choices like GPU, algorithm, and batch size are component parameters. If omitted, defaults are used.

In [4]:
input_msg_3d = Message(
    fields={"stored data (B)": 1000.0},
    properties={
        "resolution (n,n)": (64, 64),
        "bitdepth (n)": 16,
        "stored images (n)": 1000,
    },
)

analysis_ptychi = Component(
    "Pty-Chi analysis",
    [InputMessage(), PhaseReconstruction3D_model()],
    {
        "input message (Message)": input_msg_3d,
        "gpu (str)": "A100",
        "algorithm (str)": "dm",
        "batch size (n)": 1000,
        "iterations (n)": 500,
        "xy images (n,n)": (32, 32),
        "overlap (%)": 0.40,
    },
)

g_ptychi = ExecutionGraph("Pty-Chi reconstruction performance", [analysis_ptychi], [])()
pd.Series(g_ptychi.root_node.properties)

phase reconstruction ops (n,n)                                                         737280000000.0
phase reconstruction energy (J)                                                           6986.303811
phase reconstruction power (W)                                                             265.957557
phase reconstruction images per joule (images/J)                                             0.143137
phase reconstruction io latency (s)                                                          0.412566
phase reconstruction compute latency (s)                                                    25.855925
phase reconstruction io energy (J)                                                          13.368183
phase reconstruction compute energy (J)                                                   6972.935628
phase reconstruction metadata                       {'backend': 'ptychi', 'gpu': 'A100', 'algorith...
dtype: object

## 3D ptychography reconstruction with PtychoPINN

PtychoPINN is a separate backend. It does not take a Pty-Chi algorithm parameter. By default it estimates grouped samples as 0.78 * stored images.

In [5]:
pinn_model = PtychoPINNResourceModel.from_bundled_data()

analysis_pinn = Component(
    "PtychoPINN analysis",
    [InputMessage(), PhaseReconstruction3D_model(resource_model=pinn_model)],
    {
        "input message (Message)": input_msg_3d,
        "gpu (str)": "A100",
        "batch size (n)": 1024,
        "xy images (n,n)": (32, 32),
        "overlap (%)": 0.40,
    },
)

g_pinn = ExecutionGraph("PtychoPINN reconstruction performance", [analysis_pinn], [])()
pd.Series(g_pinn.root_node.properties)

phase reconstruction ops (n,n)                                                        1302600000000.0
phase reconstruction energy (J)                                                             16.691648
phase reconstruction power (W)                                                              33.754596
phase reconstruction images per joule (images/J)                                            59.910204
phase reconstruction io latency (s)                                                            0.1281
phase reconstruction compute latency (s)                                                       0.3664
phase reconstruction io energy (J)                                                           4.320866
phase reconstruction compute energy (J)                                                     12.370781
phase reconstruction metadata                       {'backend': 'ptychopinn', 'gpu': 'A100', 'num_...
dtype: object

## Small sweep

This compares images per joule for different GPUs and algorithms where coefficients are available.

In [ ]:
rows = []
for gpu in ["A100", "H200", "MI300X"]:
    for algorithm in ["pie", "dm", "lsqml", "bh", "ad_ptycho"]:
        est = ptychi.predict(
            num_images=1000,
            resolution=(64, 64),
            gpu=gpu,
            algorithm=algorithm,
        )
        rows.append({
            "backend": "ptychi",
            "gpu": gpu,
            "algorithm": algorithm,
            "latency_s": est.latency_s,
            "energy_j": est.energy_j,
            "images_per_joule": est.images_per_joule,
        })

for gpu in ["A100", "H200", "MI300X"]:
    est = ptychopinn.predict(num_images=1000, resolution=(64, 64), gpu=gpu)
    rows.append({
        "backend": "ptychopinn",
        "gpu": gpu,
        "algorithm": "",
        "latency_s": est.latency_s,
        "energy_j": est.energy_j,
        "images_per_joule": est.images_per_joule,
    })

pd.DataFrame(rows).sort_values("images_per_joule", ascending=False).reset_index(drop=True)